# Lecture 20 - Structuring code

**Lecture Topic**: How code grows up, from a notebook cell to a package, and the design patterns that give you and your agent a shared vocabulary for talking about structure.

## Announcements

- **M2 (design) is due next week**, Monday Nov 23. Your architecture and per-stage scaling analysis from last lecture should be useful.
- Last lecture was the shape of the *system*. Today is the shape of the *code*.
- ClassBuzz poll for today: `XXXX`

## Where we are

Module D is software engineering: the habits that make agent-built code something you can trust, extend, and hand off.

 - S21: **architecture**, the shape of the system.
 - S22 (today): **structuring code**, the shape of the code.
 - S23: **testing**, checking code you did not write.

Today's thread: structure is a cost. You add it when it buys you something, and the design patterns are how you ask an agent for the right amount.

Today:
 - From a notebook to a package
 - Interfaces, and functions vs. classes
 - Design patterns as a shared vocabulary
 - How much structure is enough

# From a notebook to a package

Code grows along a path, and each step buys you something for a bit more effort:

 - **Notebook cell**: quick, visual, great for exploring. Hard to reuse or test.
 - **Script**: runs start to finish from the terminal. Reproducible.
 - **Module**: a `.py` file of functions you can `import` from more than one place.
 - **Package**: a folder of modules with an interface, tests, and a version, that other people install.

You do not start at the end. You move one step along when something pushes you: you have copy-pasted the code twice, someone else needs it, or it needs a test.

# Interfaces and contracts

Every function has an **interface**: the inputs it expects and the output it promises. Everything else is detail the caller should not have to know. **Type hints** are a lightweight way to write that contract down, and an agent reads them as part of the spec.

In [ ]:
import pandas as pd

def tip_rate(df: pd.DataFrame, min_fare: float = 2.5) -> pd.Series:
    """Tip as a fraction of fare, for trips above `min_fare`. One value per row."""
    ok = df[df["fare_amount"] >= min_fare]
    return ok["tip_amount"] / ok["fare_amount"]

taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
print(tip_rate(taxi).head(3).round(3).tolist())

The signature and the docstring tell a caller (or an agent) everything they need: what goes in, what comes out, and the one option. Keep the interface small and honest and you can change the insides freely.

## Functions vs. classes

A **function** transforms inputs to outputs. A **class** bundles data together with the behaviour that acts on it. Reach for a class when state and behaviour genuinely belong together (a connection you open, use, and close; a model you fit and then predict with). For a straight transformation, a function is lighter and easier to test.

The common mistake, and one agents make too, is wrapping a single function in a class for no reason. OOP earns its keep sometimes, not everywhere.

### Question 1

You need to convert a temperature from Celsius to Fahrenheit. Function or class?

 - A. A class `TemperatureConverter` with a `.convert()` method
 - B. A one-line function `to_fahrenheit(c)`
 - C. A class with a config object and a factory
 - D. It does not matter

**B.** There is no state to hold, so a function says everything. A class here is structure with no payoff, which is harder to read and test. The skill all lecture is spotting that difference.

# Design patterns: the catalogue

Last lecture named the idea: a **design pattern** is a named, reusable solution to a recurring design problem, and because an agent already knows them from its training data, naming one is a compact, precise instruction. "Refactor this into a strategy" hands over a whole design in three words.

Now the catalogue. You do not need all twenty-odd patterns from the 1994 "Gang of Four" book; a handful cover most of what you reach for, and each is a phrase you can hand to an agent. Here are six, starting with two you have used already without naming them.

## Strategy: swap an interchangeable behaviour

A pile of `if/elif` choosing between behaviours is the signal. The pattern pulls each behaviour out so you can swap it without touching the rest.

In [ ]:
# before: the choices are tangled into one function
def summarise_tangled(values, how):
    if how == "mean":
        return sum(values) / len(values)
    elif how == "max":
        return max(values)
    elif how == "range":
        return max(values) - min(values)
    else:
        raise ValueError(how)

# after: each behaviour is a strategy you look up
strategies = {
    "mean":  lambda v: sum(v) / len(v),
    "max":   max,
    "range": lambda v: max(v) - min(v)
}

def summarise(values, how):
    return strategies[how](values)

nums = [2, 4, 4, 8]
print({how: summarise(nums, how) for how in strategies})

Adding a summary is now one line in the table, and "add a median strategy" has an obvious home. In Python a strategy is often just a function you pass in, because functions are first-class. You have been using this pattern without naming it.

## Decorator: wrap something to add behaviour

You have definitely used this one. The `@` syntax is Python's built-in support for the Decorator pattern: wrap a function to add behaviour (timing, caching, logging, retries) without editing the function itself.

In [ ]:
import time
import functools

def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        print(f"  {fn.__name__} took {(time.perf_counter() - start) * 1000:.1f} ms")
        return result
    return wrapper

@timed
def total(n):
    return sum(range(n))

print("result:", total(1_000_000))

`functools.lru_cache` is a caching decorator you get for free, and the same idea is behind a retry wrapper or a `@st.cache_data` in a dashboard. "Add a retry decorator to this" is a one-line request an agent will understand exactly.

## Factory: build the right thing from an input

A factory hides *which* concrete object to build, and *how* to set it up, behind one call. You hand it a file path and get back a loader that is ready to use, whether that needed pandas or xarray underneath. This is the natural partner to last lecture's data stores.

In [ ]:
import pandas as pd
import xarray as xr

class CsvLoader:
    def __init__(self, path):
        self.path = path
    def load(self):
        return pd.read_csv(self.path)

class NetCDFLoader:
    def __init__(self, path, variable):
        self.path, self.variable = path, variable     # NetCDF also needs a variable name
    def load(self):
        ds = xr.open_dataset(self.path)
        return ds[self.variable].to_dataframe().reset_index()

def make_loader(path):
    """Factory: build the right loader for a path, set up the way it needs."""
    if path.endswith(".csv"):
        return CsvLoader(path)
    if path.endswith(".nc"):
        return NetCDFLoader(path, variable="tas")
    raise ValueError(f"no loader for {path}")

for path in ["../data/2019_Yellow_Taxi_Trip_Data.csv", "../data/global_temparature.nc"]:
    loader = make_loader(path)
    print(f"{type(loader).__name__:13} -> {loader.load().shape}")

The loop at the bottom is the same for both files. It never imports xarray or knows that a NetCDF file needs a variable name, that set-up lives in one place. Adding Parquet is one more class and one more line in `make_loader`.

Note, this can look like Strategy, since both choose between alternatives. The difference is what gets chosen and who chooses. With Strategy, the caller picks a *behaviour* and passes it in ("use this distance metric"). With a factory, the caller asks for a *thing*, and the factory decides which class to build from the input (here the file name) and how to set it up.

## Adapter: make a mismatched interface fit

An adapter wraps something whose interface does not match what your code expects, so the two can work together. It is how you glue a third-party library, or an agent's function, to your own interface without rewriting either.

In [ ]:
# our loaders (from the factory) have a .load() method,
# but pandas gives us a bare function. an adapter bridges them.
class FunctionLoader:
    """Adapter: make a plain function look like our loader interface."""
    def __init__(self, fn, path):
        self.fn, self.path = fn, path
    def load(self):
        return self.fn(self.path)

loader = FunctionLoader(pd.read_csv, "../data/2019_Yellow_Taxi_Trip_Data.csv")
print("via .load():", loader.load().shape)

Now the function fits anywhere the code expects a `.load()`. "Write an adapter so this fits our loader interface" says exactly what you want.

## Observer: let parts react to events

Observer lets other parts of the code react when something happens, without the code where it happens knowing who is listening. Let's start with a training loop where every new request has meant another edit.

In [ ]:
history = []

def train(n_epochs):
    for epoch in range(n_epochs):
        loss = 1 / (epoch + 1)                 # stand-in for real training
        print(f"epoch {epoch}: loss {loss:.3f}")
        history.append(loss)                   # someone asked for this
        if loss < 0.3:
            print("  ALERT: loss below 0.3")   # and then this

train(5)

Each new request (keep a history, send an alert) means editing `train()` again. Instead, `train()` can just announce each epoch and let the listeners decide what to do.

In [ ]:
def train(n_epochs, callbacks=()):
    for epoch in range(n_epochs):
        loss = 1 / (epoch + 1)
        for cb in callbacks:
            cb(epoch, loss)                    # tell every listener

def alert(epoch, loss):
    if loss < 0.3:
        print("  ALERT: loss below 0.3")

history = []
train(5, callbacks=[
    lambda epoch, loss: print(f"epoch {epoch}: loss {loss:.3f}"),
    lambda epoch, loss: history.append(loss),
    alert,
])
print("history:", [round(l, 2) for l in history])

Now `train()` does not change when someone wants another side effect, they just add a callback to the list. This is how Keras callbacks work (`model.fit(..., callbacks=[EarlyStopping()])`), and logging handlers and GUI button clicks work the same way.

Note, this looks a bit like Strategy, since both pass functions in. The difference is that a strategy is *one* function that decides how the work is done, so it changes the result. Observers can be *many* functions that are told something happened, and they do not change the result.

## Pipeline: compose stages into one flow

The pipeline from last lecture, written small: a list of stage functions, each one's output feeding the next. Every stage is a plain function you can test on its own.

In [ ]:
def pipeline(*stages):
    """Compose stage functions: run them left to right."""
    def run(data):
        for stage in stages:
            data = stage(data)
        return data
    return run

# small, general stages: none of them knows about taxis
def at_least(col, value):
    return lambda df: df[df[col] >= value]

def add_ratio(name, num, den):
    return lambda df: df.assign(**{name: df[num] / df[den]})

def top(n, by, cols):
    return lambda df: df.nlargest(n, by)[cols]

taxi_tips = pipeline(
    at_least("fare_amount", 2.5),
    add_ratio("tip_rate", "tip_amount", "fare_amount"),
    top(3, "tip_rate", ["pulocationid", "tip_rate"]),
)
taxi = make_loader("../data/2019_Yellow_Taxi_Trip_Data.csv").load()
print(taxi_tips(taxi).round(3).to_string(index=False))

Nothing in `pipeline`, `at_least` or `top` mentions taxis. So let's point the same pieces at completely different data, the climate projections, and find the places that warm the most by the end of the century.

In [ ]:
def mean_by(keys, col):
    return lambda df: df.groupby(keys, as_index=False)[col].mean()

late_warming = pipeline(
    at_least("time", 2081),                    # reused
    mean_by(["lat", "lon"], "tas"),            # one new stage
    top(3, "tas", ["lat", "lon", "tas"]),      # reused
)
climate = make_loader("../data/global_temparature.nc").load()
print(late_warming(climate).round(2).to_string(index=False))

Same `pipeline`, two of the three stages reused unchanged, and one new stage. (The answer is the Canadian Arctic, which warms much faster than the global mean.)

Because the stages are general, you can reuse them on new data, test each one on a three-row DataFrame, and cache or re-run one step at a time. "Make this a pipeline of small, general stages" is the instruction that gets you there.

## Two more, by name

A couple you will hear that do not need a demo:

| Pattern | What it does | What you would say to the agent |
|---|---|---|
| **Facade** | a simple front over a complicated subsystem | "give me a facade so callers do not touch the internals" |
| **Singleton** | one shared instance for the whole program | often an **anti-pattern**: it is global state in disguise, and hard to test |

You already meet patterns in the wild without the names: a context manager (`with`) manages a resource, pandas method chaining is a fluent interface, and `sklearn`'s pipeline is the pipeline pattern. Naming the one you want turns a paragraph of explanation into a few words.

### Question 2

You want to try three different distance metrics in a nearest-neighbour search, switching between them without touching the search code. Which pattern are you reaching for?

 - A. Singleton
 - B. Strategy
 - C. Adapter
 - D. Facade

**B, Strategy.** The distance metric is an interchangeable behaviour, so you pass it in as a strategy and the search code never changes. "Make the distance metric a strategy" is the whole instruction.

# How much structure is enough

A named pattern is leverage, and like any leverage it can be over-used. The failure mode is the opposite of a tangled script: a prototype buried under abstractions for flexibility nobody asked for. Agents do this happily if you let them, because more structure looks more "professional".

## One prototype, two change requests

A script reads a CSV and emails a summary. Two requests arrive:

 - A: also read Parquet.
 - B: also post to Slack instead of email.

**Under-structured** (one long function): each request means surgery on the one function, and the two changes tangle together.

**Over-structured** (a plugin system built up front): you guessed the extension points before you knew the changes, and the guesses rarely match what actually arrives.

**Right-structured**: factor the two things that actually changed. The input format becomes a **factory** (CSV or Parquet), the output channel becomes a **strategy** (email or Slack), and nothing else moves.

The rule of thumb: structure for the change you can already see coming, not for every change you can imagine. When an agent hands you a design, the question is as much "is this too much?" as "is this enough?". You supply that judgment; the agent supplies the pattern.

# Recap

 - Code grows **notebook -> script -> module -> package**; move one step along when something pushes you, not before.
 - Keep **interfaces** small and honest (a signature, a docstring, type hints); then you can change the insides freely.
 - Use a **class** when state and behaviour belong together, a **function** otherwise.
 - **Design patterns** are a shared vocabulary, the third zoom level after architecture and paradigm. Six worth knowing (strategy, decorator, factory, adapter, observer, pipeline), each a compact instruction an agent already understands.
 - **How much structure** is itself the judgment: structure for the change you can see coming, and watch for an agent over-building.

**Next time (S23): testing and code review** - checking code you did not write, and reviewing an agent's pull request.